In [8]:
from langgraph.graph import StateGraph , START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
llm=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [4]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanation:str

In [5]:
def generate_joke(state:JokeState):
    prompt=f"generate a joke on topic {state['topic']} "
    response=llm.invoke(prompt).content

    return {'joke':response}

In [6]:
def  generate_explanation(state:JokeState):
    prompt=f'write an explanation for the joke {state['joke']}'
    response=llm.invoke(prompt).content

    return {'explanation':response}

In [10]:
graph=StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer=InMemorySaver()
workflow=graph.compile(checkpointer=checkpointer)



In [12]:
config1={'configurable':
            {'thread_id':'1'}
        }
intial_state={
    'topic':'pizza'
}
workflow.invoke(intial_state,config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza chef go broke?\n\nBecause he only made crust-worthy jokes!',
 'explanation': 'This joke is a pun that plays on the phrase "crust-worthy" and its similarity to "trustworthy."\n\nHere\'s the breakdown:\n\n1.  **"Crust-worthy" (literal meaning):** In the context of a pizza chef, "crust-worthy" literally refers to the crust of a pizza.\n2.  **"Crust-worthy" (pun meaning):** It sounds very similar to "trustworthy."\n    *   **Trustworthy:** Means reliable, honest, deserving of trust. If something is trustworthy, it\'s generally good and dependable.\n    *   **Crust-worthy (as a pun):** By saying his *jokes* were only "crust-worthy," the joke implies they were *not* very good. They were perhaps the bare minimum, or even the least desirable part, much like some people might leave the crust of a pizza. It suggests his jokes were lame, unfunny, or just barely acceptable – certainly not "trustworthy" in terms of delivering a good laugh.\n3.  **"Went

In [13]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza chef go broke?\n\nBecause he only made crust-worthy jokes!', 'explanation': 'This joke is a pun that plays on the phrase "crust-worthy" and its similarity to "trustworthy."\n\nHere\'s the breakdown:\n\n1.  **"Crust-worthy" (literal meaning):** In the context of a pizza chef, "crust-worthy" literally refers to the crust of a pizza.\n2.  **"Crust-worthy" (pun meaning):** It sounds very similar to "trustworthy."\n    *   **Trustworthy:** Means reliable, honest, deserving of trust. If something is trustworthy, it\'s generally good and dependable.\n    *   **Crust-worthy (as a pun):** By saying his *jokes* were only "crust-worthy," the joke implies they were *not* very good. They were perhaps the bare minimum, or even the least desirable part, much like some people might leave the crust of a pizza. It suggests his jokes were lame, unfunny, or just barely acceptable – certainly not "trustworthy" in terms of delivering a good

In [ ]:
config2={
    'configurable':{
        'thread_id':'2'
    }
}
workflow.invoke({'topic':'pasta'},config=config2)

{'topic': 'pasta',
 'joke': 'Why did the pasta get arrested?\n\nBecause it was an **impasta**!',
 'explanation': 'This is a classic **pun**!\n\nHere\'s the breakdown:\n\n1.  **Imposter (pronounced im-POS-ter):** An imposter is someone who pretends to be someone else in order to deceive others, often for fraudulent or illegal purposes. Committing fraud or deception can definitely lead to arrest.\n\n2.  **Impasta (pronounced im-PAS-tuh):** This is a made-up word for the joke. It sounds almost exactly like "imposter," but it cleverly incorporates the word "pasta" (the food) into it.\n\n**The Joke:**\n\nThe humor comes from the unexpected twist. You expect a reason related to actual pasta (like maybe it stole sauce, or was involved in a "noodle-ing" incident), but instead, you get a clever play on words that sounds like a legitimate reason for arrest ("imposter").\n\nSo, the pasta got arrested because it was an "impasta" – a play on words suggesting it was an "imposter"!'}